# 初始化 client（连接 DeepSeek API）

In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import numpy as np

load_dotenv()
api_key = os.getenv("DEEPSEEK_API_KEY")

#Embedding ues alibaba
embedding_client = OpenAI(
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1"
)

# Chatting use DeepSeek
chat_client = OpenAI(
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url="https://api.deepseek.com"
)

# tool描述，给LLM看的

In [2]:

tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "查询指定城市的当前天气信息",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "城市名称，例如 北京、上海、杭州"
                    }
                },
                "required": ["city"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_time",
            "description": "查询指定城市当地的时间",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "城市名称，例如 北京、上海、杭州"
                    }
                },
                "required": ["city"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "search_my_notes",
            "description": "在用户的个人笔记库中检索相关内容。适用于:用户询问自己的健身计划、电脑配置、学习笔记、个人记录等私人信息时使用。如果是天气、时间这类公共信息查询,不要用这个工具。",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "要在笔记中搜索的内容,应该是用户问题的核心关键词,例如 '增肌早餐'、'电脑配置'、'RAG 流程'"
                    }
                },
                "required": ["query"]
            }
        }
    }
]

# tool的执行函数，返回的结果给LLM看

In [3]:
import json
def get_weather(city: str) -> str:
    fake_data = {
        "北京": {"temp": 18, "condition": "晴"},
        "上海": {"temp": 22, "condition": "小雨"},
        "杭州": {"temp": 20, "condition": "多云"},
    }
    if city in fake_data:
        return json.dumps(fake_data[city], ensure_ascii=False)
    else:
        return json.dumps({"error": f"没有 {city} 的数据"}, ensure_ascii=False)

# 测一下函数本身
print(get_weather("杭州"))
print(get_weather("纽约"))

def get_time(city: str) -> str:
    fake_times = {
        "北京": "14:30",
        "上海": "14:30",
        "杭州": "14:30",
        "纽约": "02:30",
    }
    return json.dumps({"time": fake_times.get(city, "未知")}, ensure_ascii=False)

# 测一下函数本身能用
print(get_time("杭州"))



def search_my_notes(query :str) -> str:
    relevant_notes=retrieve(query, top_k=3)
    if not relevant_notes or relevant_notes[0]["score"]<0.3 :
        return "没找到相关笔记内容"
    result=""
    for i, r in enumerate(relevant_notes) :
        result+=f"第{i+1}个笔记的相关度为:{r["score"]}\n容为:{r["text"]}\n\n"
    return result.strip()

{"temp": 20, "condition": "多云"}
{"error": "没有 纽约 的数据"}
{"time": "14:30"}


In [7]:
#serach_my_notes的辅助函数
def cosine_similarity(v1, v2):
    """算两个向量的余弦相似度"""
    v1 = np.array(v1)
    v2 = np.array(v2)
    return np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))

def retrieve(query: str, top_k: int = 3) ->list:
    """
    根据用户问题,从知识库找最相关的 top_k 条
    返回:[{text, score}, ...]
    """
    # 1.用户问题也embedding化
    resp=embedding_client.embeddings.create(
        model="text-embedding-v3",
        input=query
    )
    query_vector = resp.data[0].embedding
    # 2. 算 query 和每条笔记的相似度
    scored_notes=[]
    for item in knowledge_base:
        score=cosine_similarity(query_vector,item["vector"])
        scored_notes.append({
            "text":item["text"],
            "score":score
        })
    scored_notes.sort(key=lambda x: x["score"], reverse=True)
    return scored_notes[:top_k]

In [4]:
#初始化向量库
from notes import notes
print(f"{len(notes)}")
all_vectors=[]
for i in range(0,len(notes),10) :
    mind=notes[i:i+10]
    resp=embedding_client.embeddings.create(
        model="text-embedding-v3",
        input=mind
    )
    all_vectors.extend(item.embedding for item in resp.data)
knowledge_base=[]
for i in range(len(notes)) :
    knowledge_base.append({
        "text": notes[i],
        "vector": all_vectors[i]
    })

7


In [ ]:
#用户给出问题
questions=["怎么练腿",
           "杭州今天什么天气",
           "我的电脑什么配置",
           "巴黎什么天气",
           "增肌早餐",
           "巴黎首都",
           "杭州几点 + embedding",
           "能不能给我安排一周的健身计划"]

#agent流程
def run_agent(question) :
    messages = [
        {"role": "user", "content": question}
    ]
    #第一次API调用
    resp = chat_client.chat.completions.create(
        model="deepseek-chat",
        messages=messages,
        tools=tools
    )
    #将LLM返回的调用工具请求添加到LLM的“记忆”中
    assistant_message=resp.choices[0].message
    messages.append(assistant_message)
    #创造一个工具对照表，快速查询获得工具
    function_registry = {
        "get_time" : get_time ,
        "get_weather" : get_weather,
        "search_my_notes": search_my_notes
    }
    if assistant_message.tool_calls :
        #将LLM需要的工具依次添加到他的“记忆”里
        for tc in resp.choices[0].message.tool_calls:
            tool_call = tc
            function_name = tc.function.name
            function_args = json.loads(tc.function.arguments)
            messages.append({
            "role": "tool",
            "tool_call_id": tc.id,
            "content": function_registry[function_name](**function_args)
            
        })
        #第二次API调用，LLM给出答案
        final_resp = chat_client.chat.completions.create(
            model="deepseek-chat",
            messages=messages,
            tools=tools
        )

        return final_resp.choices[0].message.content
    else :
        return assistant_message.content

In [18]:
for question in questions :
    print(f"\n{'='*60}")
    print(f"问题: {question}")
    print(f"{'='*60}")
    print(run_agent(question))


问题: 怎么练腿
找到了！你的笔记里有详细的 **腿部训练计划**，我来帮你整理一下：

---

## 🦵 练腿训练方案

### 1️⃣ 深蹲 或 哈克深蹲
- 作为主力动作

### 2️⃣ 保加利亚分腿蹲 3×8
- 身体向前倾斜 **15~30度**
- 下放时：**髋部向后下方推**，膝盖跟随但不过度前移，前小腿保持接近垂直
- 最低点：前大腿与地面平行，臀和腘绳肌有拉伸感
- 发力：**前脚跟踩地**，臀部带动髋伸回到起始位

### 3️⃣ 罗马尼亚硬拉
- 小腿始终保持垂直地面，腰背不可弯
- 杠铃始终不超过脚尖
- **髋关节向后**，不要过度挺胸，保持身体直立

### 4️⃣ 臀推 3×8
- 杠铃放在大腿根部
- 推起时小腿垂直地面
- 上背1/3或肩胛骨下端靠在凳子上

### 5️⃣ 单腿RLD 或 北欧腿弯举 3×8

---

### 📊 你的历史记录参考
| 动作 | 重量/次数 |
|------|----------|
| 罗马尼亚硬拉 | 5.26 → 60kg × 10次 |
| 保加利亚分腿蹲 | 5.22/5kg×12 → 5.26/12.5kg |
| 臀推 | 5.22/20kg×12 → 5.26/60kg×8 |

---

**建议：** 练腿前一定要充分热身（特别是膝盖和髋关节），训练后记得拉伸股四头肌和腘绳肌。你想了解某个动作的更多细节吗？😊

问题: 杭州今天什么天气
杭州今天的天气是 **多云**，气温约为 **20°C**。天气不错，适合出门活动，不过早晚可能会有些微凉，建议根据体感适当增减衣物哦！

问题: 我的电脑什么配置
根据你的笔记，你的电脑配置如下：

**主机配置**
- **CPU**: Intel i5-12400F
- **主板**: 微星 Pro H610M-S WiFi
- **显卡**: 微星 RTX 4060 万图师
- **内存**: 海盗船 DDR4 2×8GB 3200MHz（共16GB，咸鱼入手）
- **固态硬盘**: 三星 PM981 256GB（咸鱼入手）
- **移动硬盘**: 三星 T7 1TB
- **电源**: 长城 6000DS 500W
- **机箱+风扇**: 咸鱼机箱 + 5个风扇

这是一套**中端游戏配置**，总价大约 **4490元**